## 🔍 Информация о GPU

In [4]:
import torch
import sys

print("=" * 60)
print("🎮 ИНФОРМАЦИЯ О GPU И CUDA")
print("=" * 60)

print(f"\n📦 PyTorch версия: {torch.__version__}")
print(f"🔧 CUDA доступна: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"🎯 CUDA версия: {torch.version.cuda}")
    print(f"🔢 cuDNN версия: {torch.backends.cudnn.version()}")
    print(f"📊 Количество GPU: {torch.cuda.device_count()}")
    
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"\n🖥️  GPU {i}: {props.name}")
        print(f"   💾 Память: {props.total_memory / 1024**3:.1f} GB")
        print(f"   🧮 Compute Capability: {props.major}.{props.minor}")
        print(f"   ⚡ Multi-processors: {props.multi_processor_count}")
else:
    print("\n⚠️  GPU не обнаружена. Проверьте:")
    print("   1. Установлены ли драйверы NVIDIA")
    print("   2. Запущен ли контейнер с --gpus all")
    print("   3. Версия CUDA совместима с драйвером")

🎮 ИНФОРМАЦИЯ О GPU И CUDA

📦 PyTorch версия: 2.9.0+cu130
🔧 CUDA доступна: True
🎯 CUDA версия: 13.0
🔢 cuDNN версия: 91701
📊 Количество GPU: 1

🖥️  GPU 0: Tesla T4
   💾 Память: 14.6 GB
   🧮 Compute Capability: 7.5
   ⚡ Multi-processors: 40


## 🧪 Проверка nvidia-smi

In [6]:
!nvidia-smi


Thu Apr  2 17:47:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.95.05              Driver Version: 580.95.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:05.0 Off |                    0 |
| N/A   42C    P0             26W /   70W |    1277MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [28]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0))


True
1
Tesla T4


## ⚡ Сравнение CPU vs GPU: Умножение матриц

In [21]:
import torch
import time

def benchmark_matmul(size, device, iterations=10):
    """Бенчмарк умножения матриц"""
    a = torch.randn(size, size, device=device)
    b = torch.randn(size, size, device=device)
    
    # Прогрев
    for _ in range(3):
        _ = torch.matmul(a, b)
    
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    # Измерение
    start = time.time()
    for _ in range(iterations):
        c = torch.matmul(a, b)
    
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    elapsed = time.time() - start
    return elapsed / iterations

print("⚡ БЕНЧМАРК: УМНОЖЕНИЕ МАТРИЦ")
print("=" * 60)

sizes = [1000, 2000, 4000, 8000]
results = []

for size in sizes:
    cpu_time = benchmark_matmul(size, torch.device('cpu'))
    
    if torch.cuda.is_available():
        gpu_time = benchmark_matmul(size, torch.device('cuda'))
        speedup = cpu_time / gpu_time
        print(f"📊 {size}x{size}: CPU={cpu_time:.4f}s, GPU={gpu_time:.4f}s, Speedup={speedup:.1f}x")
        results.append((size, cpu_time, gpu_time, speedup))
    else:
        print(f"📊 {size}x{size}: CPU={cpu_time:.4f}s (GPU недоступен)")

if torch.cuda.is_available() and results:
    avg_speedup = sum(r[3] for r in results) / len(results)
    print(f"\n🚀 Средний speedup GPU: {avg_speedup:.1f}x")

⚡ БЕНЧМАРК: УМНОЖЕНИЕ МАТРИЦ


📊 1000x1000: CPU=0.0168s, GPU=0.0008s, Speedup=21.4x
📊 2000x2000: CPU=0.2408s, GPU=0.0059s, Speedup=41.0x


KeyboardInterrupt: 

## 🧠 Тест: Тренировка нейросети на GPU

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import time

# Определяем устройство
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🎯 Используем устройство: {device}")

# Создаём синтетический датасет
n_samples = 50000
n_features = 100
n_classes = 10

X = torch.randn(n_samples, n_features)
y = torch.randint(0, n_classes, (n_samples,))

dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=256, shuffle=True)

print(f"📊 Датасет: {n_samples:,} samples, {n_features} features, {n_classes} classes")

🎯 Используем устройство: cuda
📊 Датасет: 50,000 samples, 100 features, 10 classes


In [5]:
# Определяем модель
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_dim):
        super().__init__()
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.2)
            ])
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, output_dim))
        self.model = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.model(x)

# Создаём модель
model = MLP(
    input_dim=n_features,
    hidden_dims=[512, 256, 128],
    output_dim=n_classes
).to(device)

# Подсчёт параметров
n_params = sum(p.numel() for p in model.parameters())
print(f"🧠 Модель: MLP с {n_params:,} параметрами")
print(f"📍 Модель на устройстве: {next(model.parameters()).device}")

🧠 Модель: MLP с 219,018 параметрами
📍 Модель на устройстве: cuda:0


In [6]:
# Тренировка
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001)

n_epochs = 10
print(f"\n🚀 ТРЕНИРОВКА ({n_epochs} эпох)")
print("=" * 60)

start_total = time.time()

for epoch in range(n_epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    start_epoch = time.time()
    
    for batch_X, batch_y in dataloader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += batch_y.size(0)
        correct += predicted.eq(batch_y).sum().item()
    
    epoch_time = time.time() - start_epoch
    accuracy = 100 * correct / total
    avg_loss = total_loss / len(dataloader)
    
    print(f"Epoch {epoch+1:2d}/{n_epochs}: Loss={avg_loss:.4f}, Acc={accuracy:.2f}%, Time={epoch_time:.2f}s")

total_time = time.time() - start_total
print(f"\n⏱️  Общее время тренировки: {total_time:.2f}s")
print(f"📈 Среднее время эпохи: {total_time/n_epochs:.2f}s")


🚀 ТРЕНИРОВКА (10 эпох)
Epoch  1/10: Loss=2.3405, Acc=9.67%, Time=1.82s
Epoch  2/10: Loss=2.2992, Acc=12.25%, Time=1.08s
Epoch  3/10: Loss=2.2860, Acc=13.29%, Time=1.24s
Epoch  4/10: Loss=2.2750, Acc=14.34%, Time=1.20s
Epoch  5/10: Loss=2.2625, Acc=15.52%, Time=1.01s
Epoch  6/10: Loss=2.2481, Acc=16.65%, Time=1.03s
Epoch  7/10: Loss=2.2332, Acc=17.57%, Time=1.15s
Epoch  8/10: Loss=2.2140, Acc=18.35%, Time=1.06s
Epoch  9/10: Loss=2.1953, Acc=19.52%, Time=0.96s
Epoch 10/10: Loss=2.1752, Acc=20.60%, Time=1.00s

⏱️  Общее время тренировки: 11.55s
📈 Среднее время эпохи: 1.16s


## 💾 Мониторинг памяти GPU

In [7]:
if torch.cuda.is_available():
    print("💾 ИСПОЛЬЗОВАНИЕ ПАМЯТИ GPU")
    print("=" * 60)
    
    for i in range(torch.cuda.device_count()):
        allocated = torch.cuda.memory_allocated(i) / 1024**3
        reserved = torch.cuda.memory_reserved(i) / 1024**3
        total = torch.cuda.get_device_properties(i).total_memory / 1024**3
        
        print(f"\n🖥️  GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"   📊 Выделено: {allocated:.2f} GB")
        print(f"   📦 Зарезервировано: {reserved:.2f} GB")
        print(f"   💾 Всего: {total:.2f} GB")
        print(f"   📈 Использование: {100*allocated/total:.1f}%")
    
    # Очистка памяти
    torch.cuda.empty_cache()
    print("\n🧹 Кэш GPU очищен")
else:
    print("⚠️ GPU недоступен")

💾 ИСПОЛЬЗОВАНИЕ ПАМЯТИ GPU

🖥️  GPU 0: NVIDIA A10
   📊 Выделено: 0.02 GB
   📦 Зарезервировано: 1.30 GB
   💾 Всего: 22.06 GB
   📈 Использование: 0.1%

🧹 Кэш GPU очищен


## 🔥 Стресс-тест GPU (опционально)

In [ ]:
# ⚠️ Этот тест интенсивно использует GPU
# Раскомментируйте для запуска

# if torch.cuda.is_available():
#     print("🔥 СТРЕСС-ТЕСТ GPU")
#     print("=" * 60)
#     
#     size = 10000
#     device = torch.device('cuda')
#     
#     print(f"📊 Создаём матрицы {size}x{size}...")
#     a = torch.randn(size, size, device=device)
#     b = torch.randn(size, size, device=device)
#     
#     print("🔄 Выполняем 100 операций умножения...")
#     start = time.time()
#     for i in range(100):
#         c = torch.matmul(a, b)
#         if (i + 1) % 20 == 0:
#             torch.cuda.synchronize()
#             print(f"   Прогресс: {i+1}/100")
#     
#     torch.cuda.synchronize()
#     elapsed = time.time() - start
#     
#     ops_per_sec = 100 / elapsed
#     tflops = (2 * size**3 * 100) / elapsed / 1e12
#     
#     print(f"\n⏱️  Время: {elapsed:.2f}s")
#     print(f"🚀 Операций/сек: {ops_per_sec:.1f}")
#     print(f"💪 Производительность: ~{tflops:.1f} TFLOPS")
#     
#     del a, b, c
#     torch.cuda.empty_cache()

print("💡 Раскомментируйте код выше для запуска стресс-теста")

## ✅ Итоги GPU тестирования

### Если всё работает:
- ✅ `torch.cuda.is_available()` возвращает `True`
- ✅ GPU ускорение 10-100x на больших матрицах
- ✅ Модель тренируется на GPU без ошибок
- ✅ nvidia-smi показывает использование GPU

### Если GPU не работает:
1. Проверьте, что контейнер запущен с `--gpus all`
2. Убедитесь, что драйвер NVIDIA >= 570 (для CUDA 13.0)
3. Проверьте `nvidia-smi` на хосте

---

**Готово к работе!** 🚀 Теперь можете использовать GPU для своих ML проектов.